# Diabetes Disease Prediction — EDA & Modeling

**Dataset:** Pima Indians Diabetes Dataset  
**Task:** Binary Classification — Predict onset of diabetes  
**Author:** Yash  

---

### Notebook Structure
1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Preprocessing
4. Baseline Model Training & Evaluation
5. Hyperparameter Tuning with GridSearchCV
6. Final Model Evaluation
7. Feature Importance
8. Conclusions

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
CV_FOLDS = 5
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

In [ ]:
df = pd.read_csv('../data/diabetes.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

## 2. Exploratory Data Analysis

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['Outcome'].value_counts()
ax.bar(['Non-Diabetic (0)', 'Diabetic (1)'], counts.values,
       color=['steelblue', 'salmon'], edgecolor='white')
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontsize=11)
ax.set_title('Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../images/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Feature distributions by outcome
features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, feat in zip(axes.flatten(), features):
    for outcome, color, label in [(0, 'steelblue', 'Non-Diabetic'), (1, 'salmon', 'Diabetic')]:
        ax.hist(df[df['Outcome'] == outcome][feat], bins=20, alpha=0.6,
                color=color, label=label, edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.legend(fontsize=8)
fig.suptitle('Feature Distributions by Outcome', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../images/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(df.corr(), dtype=bool))
sns.heatmap(df.corr(), mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Zero-value counts (physiologically impossible → treated as missing)
ZERO_AS_NULL = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('Zero counts (treated as missing):')
print((df[ZERO_AS_NULL] == 0).sum())

## 3. Preprocessing

In [ ]:
df_clean = df.copy()
df_clean[ZERO_AS_NULL] = df_clean[ZERO_AS_NULL].replace(0, np.nan)
df_clean[ZERO_AS_NULL] = df_clean[ZERO_AS_NULL].fillna(df_clean[ZERO_AS_NULL].median())
print('Missing values after imputation:', df_clean.isnull().sum().sum())

X = df_clean[features]
y = df_clean['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train_sc.shape[0]} | Test: {X_test_sc.shape[0]}')

## 4. Baseline Model Training
Quick baseline with default hyperparameters to establish a reference point before tuning.

In [ ]:
baseline_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'XGBoost':             XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                                         random_state=RANDOM_STATE),
}

baseline_results = {}
for name, model in baseline_models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    baseline_results[name] = {
        'F1': f1_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_prob),
        'Accuracy': accuracy_score(y_test, y_pred),
    }
    print(f"{name:<25} Acc={baseline_results[name]['Accuracy']:.3f}  "
          f"F1={baseline_results[name]['F1']:.3f}  AUC={baseline_results[name]['AUC']:.3f}")

## 5. Hyperparameter Tuning with GridSearchCV

For each model we define a search grid and run `GridSearchCV` with **5-fold stratified CV**, scoring on **F1** (more informative than accuracy on imbalanced data).

In [ ]:
PARAM_GRIDS = {
    'Logistic Regression': {
        'C':       [0.01, 0.1, 1, 10, 100],
        'penalty': ['l2'],
        'solver':  ['lbfgs'],
    },
    'Random Forest': {
        'n_estimators':      [100, 200, 300],
        'max_depth':         [4, 6, 8, None],
        'min_samples_split': [2, 5, 10],
    },
    'XGBoost': {
        'n_estimators':  [100, 200],
        'max_depth':     [3, 5, 7],
        'learning_rate': [0.05, 0.1, 0.2],
    },
}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

tuned_models = {}
grid_searches = {}

for name, base_model in baseline_models.items():
    print(f'\nTuning {name}...')
    gs = GridSearchCV(
        estimator=base_model,
        param_grid=PARAM_GRIDS[name],
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    gs.fit(X_train_sc, y_train)
    tuned_models[name]  = gs.best_estimator_
    grid_searches[name] = gs
    print(f'  Best params : {gs.best_params_}')
    print(f'  Best CV F1  : {gs.best_score_:.4f}')

In [ ]:
# GridSearchCV heatmap — Random Forest: n_estimators × max_depth
rf_cv_df = pd.DataFrame(grid_searches['Random Forest'].cv_results_)
rf_cv_df['max_depth_label'] = rf_cv_df['param_max_depth'].fillna('None').astype(str)

pivot = rf_cv_df.pivot_table(
    values='mean_test_score',
    index='param_n_estimators',
    columns='max_depth_label',
    aggfunc='max',
)
col_order = sorted([c for c in pivot.columns if c != 'None'], key=int) + ['None']
pivot = pivot[[c for c in col_order if c in pivot.columns]]

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Mean CV F1'}, ax=ax)
ax.set_title('GridSearchCV — Random Forest\nn_estimators × max_depth (best F1 across min_samples_split)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('max_depth')
ax.set_ylabel('n_estimators')
plt.tight_layout()
plt.savefig('../images/gridsearch_rf_heatmap.png', dpi=150)
plt.show()

## 6. Final Tuned Model Evaluation

In [ ]:
tuned_results = {}
for name, model in tuned_models.items():
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    tuned_results[name] = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob),
        'y_pred':    y_pred,
        'y_prob':    y_prob,
    }

print(f"{'Model':<25} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'AUC':>6}")
print('-' * 60)
for name, m in tuned_results.items():
    print(f"{name:<25} {m['accuracy']:>6.3f} {m['precision']:>6.3f} "
          f"{m['recall']:>6.3f} {m['f1']:>6.3f} {m['roc_auc']:>6.3f}")

In [ ]:
# Baseline vs Tuned F1 comparison
names  = list(baseline_results.keys())
base_f1  = [baseline_results[n]['F1'] for n in names]
tuned_f1 = [tuned_results[n]['f1']    for n in names]

x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - 0.2, base_f1,  0.35, label='Baseline', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + 0.2, tuned_f1, 0.35, label='Tuned (GridSearchCV)', color='darkorange', alpha=0.9)
ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1)
ax.set_title('Baseline vs Tuned F1-Score', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../images/baseline_vs_tuned.png', dpi=150)
plt.show()

In [ ]:
# ROC Curves — tuned models
fig, ax = plt.subplots(figsize=(7, 5))
for name, m in tuned_results.items():
    fpr, tpr, _ = roc_curve(y_test, m['y_prob'])
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC={m['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], 'k--')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Tuned Models', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../images/roc_curves.png', dpi=150)
plt.show()

In [ ]:
# Confusion matrices — tuned models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, m) in zip(axes, tuned_results.items()):
    cm = confusion_matrix(y_test, m['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No DM', 'DM'], yticklabels=['No DM', 'DM'], ax=ax)
    ax.set_title(f'{name}\n(Tuned)', fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('../images/confusion_matrices_all.png', dpi=150)
plt.show()

## 7. Feature Importance (Best Tuned Model)

In [ ]:
best_name = max(tuned_results, key=lambda n: tuned_results[n]['f1'])
print(f'Best model: {best_name}')

rf_model = tuned_models['Random Forest']
feat_df = pd.DataFrame({'Feature': features, 'Importance': rf_model.feature_importances_})
feat_df = feat_df.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.barh(feat_df['Feature'], feat_df['Importance'], color='steelblue', edgecolor='white')
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.set_title('Random Forest (Tuned) — Feature Importance', fontweight='bold')
ax.set_xlabel('Importance')
ax.set_xlim(0, feat_df['Importance'].max() * 1.15)
plt.tight_layout()
plt.savefig('../images/feature_importance.png', dpi=150)
plt.show()

## 8. Conclusions

| Step | Key Finding |
|---|---|
| **Data** | 768 samples, 8 features; zero-values in 5 columns treated as missing and median-imputed |
| **EDA** | Glucose has highest correlation (0.47) with Outcome; Age and BMI also strong predictors |
| **Tuning** | GridSearchCV (5-fold, F1) improved RF F1 from baseline; best params printed above |
| **Best model** | Random Forest (tuned) — highest F1 and ROC-AUC |
| **Key predictor** | Glucose explains ~25% of RF variance — consistent with clinical evidence |

### Next Steps
- SMOTE oversampling for improved recall on diabetic class
- SHAP values for per-patient explainability
- Optuna for faster, broader hyperparameter search